In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlcoholPhenolGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolGlucuronidation, self).__init__()
        self._name = "O-Glucuronidation (Alcohols/Phenols)"
        self._target_oxygens = []
        
        # SMARTS: Επιλέγει [OX2H] που συνδέεται με άνθρακα, ο οποίος ΔΕΝ είναι καρβονύλιο
        self.PATTERN = Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")
        # Template: β-D-glucuronide με SMILES όπου ο C1 (ανωμερής) είναι το 1ο άτομο (index 0)
        # SMILES: C1([OH])O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O
        # Με αυτό το SMILES: 
        # index 0 -> ο C1 που θα ενωθεί με το υπόλοιπο μόριο
        # index 1 -> το -OH του C1 το οποίο ΠΡΕΠΕΙ να αφαιρεθεί
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AlcoholPhenolGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Το [OX2H] είναι το δεύτερο άτομο στο pattern (index 1)
            self._target_oxygens.append(match[1])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            c_sugar_idx = rdkit_mol.GetNumAtoms() 
            oh_sugar_idx = c_sugar_idx + 1 
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_sugar_idx)
            
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

alph_glucoronidation = AlcoholPhenolGlucuronidation()

# 1. Έλεγχος Φαινολικής Γλυκουρονιδίωσης (Παρακεταμόλη)
paracetamol = MolpherMol("CC(=O)Nc1ccc(O)cc1")
alph_glucoronidation.setOriginal(paracetamol)
result = alph_glucoronidation.morph()
    
print("=== TESTING ALCOHOL/PHENOL GLUCURONIDATION ===")
print(f"SOURCE: {paracetamol.getSMILES()}")
print(f"PRODUCT: {result.getSMILES() if result else 'Failed'}")
print("-" * 50)

=== TESTING ALCOHOL/PHENOL GLUCURONIDATION ===
SOURCE: CC(=O)NC1=CC=C(O)C=C1
PRODUCT: CC(=O)NC1=CC=C(OC2OC(C(=O)O)C(O)C(O)C2O)C=C1
--------------------------------------------------


In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlcoholPhenolGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolGlucuronidation, self).__init__()
        self._name = "O-Glucuronidation (Alcohols/Phenols)"
        self._target_oxygens = []
        
        # SMARTS: Επιλέγει [OX2H] που συνδέεται με άνθρακα, ο οποίος ΔΕΝ είναι καρβονύλιο
        self.PATTERN = Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")
        # Template: β-D-glucuronide με SMILES όπου ο C1 (ανωμερής) είναι το 1ο άτομο (index 0)
        # SMILES: C1([OH])O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O
        # Με αυτό το SMILES: 
        # index 0 -> ο C1 που θα ενωθεί με το υπόλοιπο μόριο
        # index 1 -> το -OH του C1 το οποίο ΠΡΕΠΕΙ να αφαιρεθεί
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AlcoholPhenolGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Το [OX2H] είναι το δεύτερο άτομο στο pattern (index 1)
            self._target_oxygens.append(match[1])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            c_sugar_idx = rdkit_mol.GetNumAtoms() 
            oh_sugar_idx = c_sugar_idx + 1 
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_sugar_idx)
            
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

alph_glucoronidation = AlcoholPhenolGlucuronidation()


class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
        
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target
            
start_mol = MolpherMol("CC(=O)Nc1ccc(O)cc1")
target_mol = MolpherMol("CC(=O)Nc1ccc(OC2OC(C(=O)O)C(O)C(O)C2O)cc1")
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (alph_glucoronidation,)

closest_info = FindClosest()

print("--- STARTING MOLPHER SEARCH TREE ---")
while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
    
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS: Target molecule reached successfully!")

--- STARTING MOLPHER SEARCH TREE ---
Generation #1
Molecules in tree: 2
Closest to target: CC(=O)NC1=CC=C(OC2OC(C(=O)O)C(O)C(O)C2O)C=C1 (Distance: 0.0000)
----------------------------------------

Search finished!
SUCCESS: Target molecule reached successfully!


In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlcoholPhenolGlucuronidation(MorphingOperator):
    def __init__(self):
        super(AlcoholPhenolGlucuronidation, self).__init__()
        self._name = "O-Glucuronidation (Alcohols/Phenols)"
        self._target_oxygens = []
        
        # SMARTS: Επιλέγει [OX2H] που συνδέεται με άνθρακα, ο οποίος ΔΕΝ είναι καρβονύλιο
        self.PATTERN = Chem.MolFromSmarts("[#6;!$(C=O);!$(C=C)][OX2H]")
        # Template: β-D-glucuronide με SMILES όπου ο C1 (ανωμερής) είναι το 1ο άτομο (index 0)
        # SMILES: C1([OH])O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O
        # Με αυτό το SMILES: 
        # index 0 -> ο C1 που θα ενωθεί με το υπόλοιπο μόριο
        # index 1 -> το -OH του C1 το οποίο ΠΡΕΠΕΙ να αφαιρεθεί
        self.GLUCURONIDE_TEMPLATE = Chem.MolFromSmiles("C1(O)O[C@H](C(=O)O)[C@@H](O)[C@H](O)[C@@H]1O")

    def setOriginal(self, mol):
        super(AlcoholPhenolGlucuronidation, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Το [OX2H] είναι το δεύτερο άτομο στο pattern (index 1)
            self._target_oxygens.append(match[1])

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)
            
            combined = Chem.CombineMols(rdkit_mol, self.GLUCURONIDE_TEMPLATE)
            rw_combined = Chem.RWMol(combined)
            c_sugar_idx = rdkit_mol.GetNumAtoms() 
            oh_sugar_idx = c_sugar_idx + 1 
            
            rw_combined.AddBond(target_o_idx, c_sugar_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_sugar_idx)
            
            new_mol = rw_combined.GetMol()
            
            for idx in [target_o_idx, c_sugar_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

alph_glucuronidation = AlcoholPhenolGlucuronidation()

traps = {
    # 1. Υπεροξείδια / Ύδροϋπεροξείδια (Έχουν -ΟΟΗ, δεν πρέπει να αντιδράσουν ως αλκοόλες)
    "Hydroperoxide Trap": "CCOO", 
    
    # 2. Ενώσεις Φωσφόρου / Σουλφονικά οξέα (Έχουν -S(=O)2-OH ή -P(=O)-OH, δεν είναι καρβοξυλικά)
    "Sulfonic Acid Trap": "CCS(=O)(=O)O",
    
    # 3. Εστέρες (Έχουν C(=O)O-C, δεν έχουν ελεύθερο -ΟΗ για να γίνει Acyl Glucuronidation)
    "Ester Trap": "CC(=O)OCC",
    
    # 4. Αμίδια / Υδροξυλαμίνες (Έχουν δεσμούς N-OH, το οξυγόνο ΔΕΝ συνδέεται με άνθρακα)
    "Hydroxylamine Trap": "CCN(C)O",
    
    # 5. Ενόλες (Ασταθείς αλκοόλες πάνω σε διπλό δεσμό, συχνά δημιουργούν σφάλματα σθένους)
    "Enol Trap": "C=CO"
}

print("\n=== RUNNING FALSE POSITIVE TRAP TESTS ===")
for name, smiles in traps.items():
    mol = MolpherMol(smiles)
    alph_glucuronidation.setOriginal(mol)
    res1 = alph_glucuronidation.morph()

    passed_alph_glucuronidation  = (res1.getSMILES() == mol.getSMILES())
    status = "SAFE (Passed)" if (passed_alph_glucuronidation) else "VULNERABLE (Failed)"
    print(f"Structure: {name} [{smiles}] -> {status}")


=== RUNNING FALSE POSITIVE TRAP TESTS ===
Structure: Hydroperoxide Trap [CCOO] -> SAFE (Passed)
Structure: Sulfonic Acid Trap [CCS(=O)(=O)O] -> SAFE (Passed)
Structure: Ester Trap [CC(=O)OCC] -> SAFE (Passed)
Structure: Hydroxylamine Trap [CCN(C)O] -> SAFE (Passed)
Structure: Enol Trap [C=CO] -> SAFE (Passed)
